# CrossHL-VLM Showcase

Notebook interface for inspecting HSI--LiDAR datasets, running CrossHL-VLM ablations, and generating analysis plots.

Core behavior:
- Cross-HL remains the classifier.
- CLIP is frozen and provides text prototypes only for semantic regularization.
- HSI and LiDAR are processed in their native modalities.
- No RGB conversion is used.


## Required Imports


In [1]:
from pathlib import Path
import csv
import json
import os
import random
import sys
import time
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score, classification_report, cohen_kappa_score, confusion_matrix,
    f1_score, log_loss, matthews_corrcoef,
)
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader, Subset

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)


PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from data import HSILidarDataset, apply_spectral_perturbation, make_fewshot_subset, make_kshot_subset
from model.CrossHL_model import CrossHL_Transformer
from prompts import PROMPT_SETS, build_text_prototypes, get_class_names, prototype_similarity_stats

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Project root:", PROJECT_ROOT)
print("Python:", sys.executable)
print("Device:", DEVICE)


Project root: c:\Users\user\Desktop\Unime - Data Analysis\Final Dissertation\CrossHL-VLM
Python: c:\Users\user\Desktop\Unime - Data Analysis\Final Dissertation\CrossHL-VLM\.venv\Scripts\python.exe
Device: cuda


## Helper Functions


In [2]:

AVAILABLE_DATASETS = ["Trento", "Houston", "MUUFL"]


def set_seed(seed=14):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def pct_tag(pct):
    return f"pct{int(round(float(pct) * 100))}"


def split_tag(split_kind, split_value):
    return f"k{int(split_value)}" if split_kind == "shot" else pct_tag(split_value)


def split_label(split_kind, split_value):
    return f"{int(split_value)}-shot" if split_kind == "shot" else f"{int(round(float(split_value) * 100))}%"


def split_sort_key(label):
    label = str(label)
    if label.endswith("-shot"):
        return (0, int(label.replace("-shot", "")))
    if label.endswith("%"):
        return (1, float(label.replace("%", "")))
    return (2, label)


def load_dataset_context(dataset):
    global DATASET, train_full, test_full, class_names, NC, NCLIDAR, CLASSES, PATCH_SIZE
    DATASET = dataset
    train_full = HSILidarDataset(PROJECT_ROOT, dataset=DATASET, split="train")
    test_full = HSILidarDataset(PROJECT_ROOT, dataset=DATASET, split="test")
    class_names = get_class_names(DATASET)
    NC = train_full.hs_image.shape[1]
    NCLIDAR = train_full.lidar_image.shape[1]
    CLASSES = len(torch.unique(train_full.lbls))
    PATCH_SIZE = train_full.hs_image.shape[-1]
    if CLASSES != len(class_names):
        raise ValueError(f"{DATASET}: labels contain {CLASSES} classes but prompts define {len(class_names)} names.")
    return train_full, test_full, class_names


def dataset_inventory(datasets=AVAILABLE_DATASETS):
    rows = []
    for dataset in datasets:
        train_ds = HSILidarDataset(PROJECT_ROOT, dataset=dataset, split="train")
        test_ds = HSILidarDataset(PROJECT_ROOT, dataset=dataset, split="test")
        labels = train_ds.lbls.detach().cpu().numpy()
        vals, counts = np.unique(labels, return_counts=True)
        rows.append({
            "dataset": dataset,
            "classes": len(vals),
            "train_samples": len(train_ds),
            "test_samples": len(test_ds),
            "hsi_bands": train_ds.hs_image.shape[1],
            "lidar_channels": train_ds.lidar_image.shape[1],
            "patch": train_ds.hs_image.shape[-1],
            "min_train_per_class": int(counts.min()),
            "max_train_per_class": int(counts.max()),
        })
    return pd.DataFrame(rows)


def make_run_dir(base="runs", name=None):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_name = name or f"notebook_{DATASET}_{timestamp}"
    run_dir = PROJECT_ROOT / base / run_name
    (run_dir / "checkpoints" / DATASET).mkdir(parents=True, exist_ok=True)
    (run_dir / "logs" / DATASET).mkdir(parents=True, exist_ok=True)
    (run_dir / "analysis").mkdir(parents=True, exist_ok=True)
    return run_dir


def append_csv_row(path, row):
    path.parent.mkdir(parents=True, exist_ok=True)
    exists = path.exists()
    with path.open("a", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(row.keys()))
        if not exists:
            writer.writeheader()
        writer.writerow(row)


def write_csv(path, rows):
    if not rows:
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


def make_balanced_eval_subset(dataset, max_samples=1024, seed=14):
    labels = dataset.lbls.detach().cpu()
    classes = torch.unique(labels).tolist()
    per_class = max(1, int(max_samples) // max(1, len(classes)))
    generator = torch.Generator().manual_seed(seed)
    selected = []
    counts = {}
    all_indices = torch.arange(len(labels))
    for class_id in classes:
        class_indices = all_indices[labels == class_id]
        n_keep = min(per_class, len(class_indices))
        perm = class_indices[torch.randperm(len(class_indices), generator=generator)]
        selected.append(perm[:n_keep])
        counts[int(class_id)] = int(n_keep)
    return Subset(dataset, torch.cat(selected).tolist()), counts


# Dataset Overview

Inspect available HSI--LiDAR datasets and class balance before launching a run.


In [3]:

DATASET = "Trento"
FM = 16
BATCH_SIZE = 64
TEST_BATCH_SIZE = 500
SEED = 14
SPLIT_SEED = 14

set_seed(SEED)
load_dataset_context(DATASET)

print("Dataset:", DATASET)
print("Train samples:", len(train_full))
print("Test samples:", len(test_full))
print("HSI:", tuple(train_full.hs_image.shape))
print("LiDAR:", tuple(train_full.lidar_image.shape))
print("Classes:", CLASSES, class_names)

display(dataset_inventory())


Dataset: Trento
Train samples: 819
Test samples: 29395
HSI: (819, 63, 11, 11)
LiDAR: (819, 1, 11, 11)
Classes: 6 ['Buildings', 'Woods', 'Roads', 'Apples', 'ground', 'Vineyard']


,dataset,classes,train_samples,test_samples,hsi_bands,lidar_channels,patch,min_train_per_class,max_train_per_class
0,Trento,6,819,29395,63,1,11,105,184
1,Houston,15,2832,12197,144,1,11,181,198
2,MUUFL,11,2683,51004,64,2,11,9,1162


In [4]:

SPLIT_KIND = "shot"  # "shot" or "pct"
INSPECT_SHOT = 15
INSPECT_PCT = 0.10

if SPLIT_KIND == "shot":
    inspect_subset, inspect_counts = make_kshot_subset(train_full, shots=INSPECT_SHOT, seed=SPLIT_SEED)
    print(f"Few-shot setting: {INSPECT_SHOT}-shot per class")
else:
    inspect_subset, inspect_counts = make_fewshot_subset(train_full, pct=INSPECT_PCT, seed=SPLIT_SEED)
    print(f"Few-shot setting: {int(INSPECT_PCT * 100)}%")

print("Subset samples:", len(inspect_subset))
print("Per-class selected samples:", {class_names[k]: v for k, v in inspect_counts.items()})


Few-shot setting: 15-shot per class
Subset samples: 90
Per-class selected samples: {'Buildings': 15, 'Woods': 15, 'Roads': 15, 'Apples': 15, 'ground': 15, 'Vineyard': 15}


# Cross-HL Model

The classifier path is the same Cross-HL logic: fused CLS feature goes to `fclayer`. The VLM branch is optional and starts after the encoder feature.


In [5]:
model = CrossHL_Transformer(
    FM=FM,
    NC=NC,
    NCLidar=NCLIDAR,
    Classes=CLASSES,
    patchsize=PATCH_SIZE,
).to(DEVICE)
model.eval()

sample_loader = DataLoader(inspect_subset, batch_size=4, shuffle=False)
hsi, lidar, labels = next(iter(sample_loader))
hsi = hsi.to(DEVICE)
lidar = lidar.to(DEVICE)
labels = labels.to(DEVICE)

with torch.no_grad():
    logits_only = model(hsi, lidar)
    logits_branch, img_embed, cls_features = model(
        hsi,
        lidar,
        return_embed=True,
        return_features=True,
    )

print("Logits only:", tuple(logits_only.shape))
print("Logits with branch:", tuple(logits_branch.shape))
print("VLM projection embedding:", tuple(img_embed.shape))
print("Raw fused CLS feature:", tuple(cls_features.shape))
print("Classifier output unchanged when branch is requested:", torch.allclose(logits_only, logits_branch, atol=1e-6))
print(model)


Logits only: (4, 6)
Logits with branch: (4, 6)
VLM projection embedding: (4, 512)
Raw fused CLS feature: (4, 64)
Classifier output unchanged when branch is requested: True
CrossHL_Transformer(
  (conv5): Sequential(
    (0): Conv3d(1, 8, kernel_size=(9, 3, 3), stride=(1, 1, 1), padding=(0, 1, 1))
    (1): BatchNorm3d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (hetconv_layer): Sequential(
    (0): HetConv(
      (groupwise_conv): Conv2d(440, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=8)
      (pointwise_conv): Conv2d(440, 64, kernel_size=(1, 1), stride=(1, 1))
    )
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (ca): Encoder(
    (layer): ModuleList(
      (0-1): 2 x SingleEncoderBlock(
        (attention_norm): LayerNorm((64,), eps=1e-06, elementwise_affine=True)
        (ffn_norm): LayerNorm((64,), eps=1e-06, elementwise_affine=True)
        (ffn): MultiLay

# Prompt Sets

Minimum ablation:
- baseline: no semantic loss
- name: class-name prompts
- spectral: physics/spectral attribute prompts

Extra multimodal ablation:
- spectral_lidar: spectral/material plus LiDAR structure/elevation cues, because Cross-HL aligns a fused HSI+LiDAR feature.


In [6]:
for mode in ["name", "spectral", "spectral_lidar"]:
    print("\n" + "=" * 100)
    print(mode.upper())
    for class_name, prompts in PROMPT_SETS[DATASET][mode].items():
        print(f"\n{class_name}")
        for prompt in prompts:
            print("  -", prompt)



NAME

Buildings
  - land cover class buildings
  - remote sensing class buildings

Woods
  - land cover class woods
  - remote sensing class woods

Roads
  - land cover class roads
  - remote sensing class roads

Apples
  - land cover class apple orchard
  - remote sensing class apple orchard

ground
  - land cover class bare ground
  - remote sensing class bare ground

Vineyard
  - land cover class vineyard
  - remote sensing class vineyard

SPECTRAL

Buildings
  - roof masonry tile concrete artificial non vegetated flat reflectance
  - concrete tile masonry roof artificial flat non vegetated reflectance

Woods
  - forest dense closed canopy high biomass continuous tree crown NIR plateau
  - dense closed forest continuous tree crowns high biomass NIR plateau

Roads
  - asphalt road black bitumen pavement low NIR linear impervious surface
  - black bitumen asphalt road linear pavement low NIR impervious surface

Apples
  - apple orchard deciduous fruit trees separated crowns grass und

# CLIP Text Encoder

Set `LOAD_CLIP=True` when running semantic experiments inside the notebook. CLIP is frozen. It only creates text prototypes.


In [7]:

LOAD_CLIP = True
CLIP_MODEL = "ViT-B-32"
CLIP_PRETRAINED = "laion2b_s34b_b79k"
CENTER_PROTOTYPES = True

clip_model = None
tokenizer = None
text_prototype_cache = {}
text_prototypes = {}


def ensure_clip_loaded():
    global clip_model, tokenizer
    if clip_model is not None and tokenizer is not None:
        return
    if not LOAD_CLIP:
        raise RuntimeError("LOAD_CLIP=False. Set LOAD_CLIP=True before semantic ablations.")
    import open_clip
    clip_model, _, _ = open_clip.create_model_and_transforms(CLIP_MODEL, pretrained=CLIP_PRETRAINED)
    tokenizer = open_clip.get_tokenizer(CLIP_MODEL)
    clip_model = clip_model.to(DEVICE).eval()
    for param in clip_model.parameters():
        param.requires_grad = False


def maybe_center_prototypes(prototypes):
    centered = prototypes - prototypes.mean(dim=0, keepdim=True)
    return F.normalize(centered, dim=-1)


def ensure_text_prototypes(dataset):
    global text_prototypes
    if dataset in text_prototype_cache:
        text_prototypes = text_prototype_cache[dataset]
        return text_prototypes
    ensure_clip_loaded()
    prototypes_by_mode = {}
    print(f"Building frozen CLIP text prototypes for {dataset}")
    for mode in ["name", "spectral", "spectral_lidar"]:
        prototypes = build_text_prototypes(clip_model, tokenizer, dataset, mode, DEVICE)
        stats = prototype_similarity_stats(prototypes)
        if CENTER_PROTOTYPES:
            prototypes = maybe_center_prototypes(prototypes)
            centered_stats = prototype_similarity_stats(prototypes)
            print(
                f"{dataset} / {mode}: raw mean/max={stats['mean_offdiag']:.3f}/{stats['max_offdiag']:.3f}; "
                f"centered mean/max={centered_stats['mean_offdiag']:.3f}/{centered_stats['max_offdiag']:.3f}"
            )
        else:
            print(f"{dataset} / {mode}: mean off-diagonal={stats['mean_offdiag']:.3f}, max={stats['max_offdiag']:.3f}")
        prototypes_by_mode[mode] = prototypes
    text_prototype_cache[dataset] = prototypes_by_mode
    text_prototypes = prototypes_by_mode
    return prototypes_by_mode


if LOAD_CLIP:
    ensure_text_prototypes(DATASET)
else:
    print("CLIP not loaded. Baseline and no-CLIP sanity cells can still run.")
    print("Set LOAD_CLIP=True before semantic ablation cells.")


c:\Users\user\Desktop\Unime - Data Analysis\Final Dissertation\CrossHL-VLM\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Building frozen CLIP text prototypes for Trento
Trento / name: raw mean/max=0.795/0.882; centered mean/max=-0.195/0.173
Trento / spectral: raw mean/max=0.525/0.674; centered mean/max=-0.199/0.163
Trento / spectral_lidar: raw mean/max=0.543/0.670; centered mean/max=-0.199/0.102


# Evaluation Function


In [8]:

def evaluate_model(model, loader, device=DEVICE, class_count=None):
    if class_count is None:
        class_count = CLASSES
    y_true = []
    y_pred = []
    model.eval()
    with torch.no_grad():
        for hsi_batch, lidar_batch, label_batch in loader:
            hsi_batch = hsi_batch.to(device)
            lidar_batch = lidar_batch.to(device)
            logits = model(hsi_batch, lidar_batch)
            pred = torch.argmax(logits, dim=1)
            y_true.append(label_batch.cpu().numpy())
            y_pred.append(pred.cpu().numpy())

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)
    conf = confusion_matrix(y_true, y_pred, labels=list(range(class_count)))
    class_totals = conf.sum(axis=1)
    per_class = np.full(class_count, np.nan, dtype=np.float64)
    valid_classes = class_totals > 0
    per_class[valid_classes] = (np.diag(conf)[valid_classes] / class_totals[valid_classes]) * 100.0
    aa = float(np.nanmean(per_class)) if np.any(valid_classes) else 0.0
    per_class = np.nan_to_num(per_class, nan=0.0)
    return {
        "oa": accuracy_score(y_true, y_pred) * 100.0,
        "aa": aa,
        "kappa": cohen_kappa_score(y_true, y_pred) * 100.0,
        "per_class": per_class,
        "confusion": conf,
    }


# Training Function

Train one experiment for one split and iteration.


In [9]:
def train_one_experiment(
    exp_name,
    prompt_mode,
    lambda_sem,
    train_loader,
    test_loader,
    run_dir,
    split_kind,
    split_value,
    iteration,
    epochs=200,
    lr=5e-4,
    weight_decay=5e-3,
    semantic_start_epoch=0,
    lambda_warmup_epochs=50,
    temperature=0.07,
    spectral_noise_std=0.0,
    spectral_gain_std=0.0,
    grad_clip=1.0,
    freeze_pct_threshold=0.0,
    save_best_by_test=False,
    eval_interval=50,
):
    set_seed(SEED + iteration)
    model = CrossHL_Transformer(
        FM=FM,
        NC=NC,
        NCLidar=NCLIDAR,
        Classes=CLASSES,
        patchsize=PATCH_SIZE,
    ).to(DEVICE)

    freeze_early = freeze_pct_threshold > 0 and split_kind == "pct" and split_value <= freeze_pct_threshold
    if freeze_early:
        model.freeze_early_layers()
        print("Early convolution layers frozen for 1% regime.")

    prototypes = None
    if prompt_mode is not None:
        if prompt_mode not in text_prototypes:
            raise RuntimeError(f"Text prototypes for {DATASET}/{prompt_mode} are missing. Set LOAD_CLIP=True and run the CLIP cell.")
        prototypes = text_prototypes[prompt_mode].to(DEVICE)

    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
        weight_decay=weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    ce_loss = nn.CrossEntropyLoss()

    tag = f"{exp_name}_{split_tag(split_kind, split_value)}_lam{lambda_sem:g}"
    best_state = None
    best_eval = None
    best_oa = -1.0
    epoch_rows = []
    start = time.time()

    for epoch in range(epochs):
        model.train()
        if epoch < semantic_start_epoch:
            lambda_scale = 0.0
        else:
            warmup_epoch = epoch - semantic_start_epoch + 1
            lambda_scale = min(1.0, float(warmup_epoch) / max(1, lambda_warmup_epochs))
        current_lambda = lambda_sem * lambda_scale
        ce_total = 0.0
        sem_total = 0.0
        total_loss = 0.0
        batches = 0

        for hsi_batch, lidar_batch, label_batch in train_loader:
            hsi_batch = hsi_batch.to(DEVICE)
            lidar_batch = lidar_batch.to(DEVICE)
            label_batch = label_batch.to(DEVICE)
            hsi_batch = apply_spectral_perturbation(hsi_batch, spectral_noise_std, spectral_gain_std)

            optimizer.zero_grad()
            if prototypes is None or current_lambda <= 0:
                logits = model(hsi_batch, lidar_batch)
                loss_sem = torch.tensor(0.0, device=DEVICE)
            else:
                logits, img_embed = model(hsi_batch, lidar_batch, return_embed=True)
                semantic_logits = (img_embed @ prototypes.T) / temperature
                loss_sem = F.cross_entropy(semantic_logits, label_batch)

            loss_ce = ce_loss(logits, label_batch)
            loss = loss_ce + current_lambda * loss_sem
            loss.backward()
            if grad_clip > 0:
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

            ce_total += loss_ce.item()
            sem_total += loss_sem.item()
            total_loss += loss.item()
            batches += 1

        scheduler.step()

        eval_result = None
        if (epoch + 1) % eval_interval == 0 or epoch == epochs - 1:
            eval_result = evaluate_model(model, test_loader)
            if save_best_by_test and eval_result["oa"] > best_oa:
                best_oa = eval_result["oa"]
                best_eval = eval_result
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            print(
                f"{tag} | iter {iteration} | epoch {epoch + 1}/{epochs} | "
                f"CE={ce_total / max(1, batches):.4f} | Sem={sem_total / max(1, batches):.4f} | "
                f"OA={eval_result['oa']:.2f}"
            )

        epoch_rows.append({
            "epoch": epoch + 1,
            "experiment": exp_name,
            "tag": tag,
            "pct": split_value if split_kind == "pct" else "",
            "shots": split_value if split_kind == "shot" else "",
            "split_label": split_label(split_kind, split_value),
            "iteration": iteration,
            "lambda_sem": current_lambda,
            "loss_ce": ce_total / max(1, batches),
            "loss_sem": sem_total / max(1, batches),
            "loss_total": total_loss / max(1, batches),
            "test_oa": "" if eval_result is None else eval_result["oa"],
            "test_aa": "" if eval_result is None else eval_result["aa"],
            "test_kappa": "" if eval_result is None else eval_result["kappa"],
        })

    if save_best_by_test and best_state is not None:
        model.load_state_dict(best_state)
        final_eval = best_eval
        selection = "best_test"
    else:
        final_eval = evaluate_model(model, test_loader)
        selection = "final_epoch"

    ckpt_path = run_dir / "checkpoints" / DATASET / f"net_params_CrossHL_{tag}_Iter{iteration}.pkl"
    ckpt_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), ckpt_path)
    write_csv(run_dir / "logs" / DATASET / f"epochs_{tag}_Iter{iteration}.csv", epoch_rows)
    np.savetxt(run_dir / "logs" / DATASET / f"confusion_{tag}_Iter{iteration}.csv", final_eval["confusion"], delimiter=",", fmt="%d")

    summary = {
        "dataset": DATASET,
        "experiment": exp_name,
        "prompt_mode": "" if prompt_mode is None else prompt_mode,
        "tag": tag,
        "pct": split_value if split_kind == "pct" else "",
        "shots": split_value if split_kind == "shot" else "",
        "pct_label": split_label(split_kind, split_value),
        "split_label": split_label(split_kind, split_value),
        "iteration": iteration,
        "lambda_sem": lambda_sem,
        "freeze_early": freeze_early,
        "selection": selection,
        "oa": final_eval["oa"],
        "aa": final_eval["aa"],
        "kappa": final_eval["kappa"],
        "seconds": time.time() - start,
        "checkpoint": str(ckpt_path),
    }
    for idx, class_name in enumerate(class_names):
        summary[f"acc_{class_name}"] = final_eval["per_class"][idx]
    append_csv_row(run_dir / "summary.csv", summary)
    print(f"FINAL {tag} Iter{iteration}: OA={summary['oa']:.2f}, AA={summary['aa']:.2f}, Kappa={summary['kappa']:.2f}")
    return summary


# Quick Notebook Sanity Test

This runs one mini optimization step only. It does not create paper results. It is here to prove that the notebook can train directly without using shell commands.


In [ ]:

RUN_MINI_STEP = False

if RUN_MINI_STEP:
    set_seed(SEED)
    mini_model = CrossHL_Transformer(FM=FM, NC=NC, NCLidar=NCLIDAR, Classes=CLASSES, patchsize=PATCH_SIZE).to(DEVICE)
    mini_model.train()
    mini_optimizer = torch.optim.Adam(mini_model.parameters(), lr=5e-4)
    mini_loader = DataLoader(inspect_subset, batch_size=4, shuffle=True)
    mini_hsi, mini_lidar, mini_labels = next(iter(mini_loader))
    mini_hsi = mini_hsi.to(DEVICE)
    mini_lidar = mini_lidar.to(DEVICE)
    mini_labels = mini_labels.to(DEVICE)
    mini_logits = mini_model(mini_hsi, mini_lidar)
    mini_loss = F.cross_entropy(mini_logits, mini_labels)
    mini_optimizer.zero_grad()
    mini_loss.backward()
    mini_optimizer.step()
    print("One notebook training step completed. Loss:", float(mini_loss.detach()))
else:
    print("Mini step skipped. Set RUN_MINI_STEP=True for a one-batch sanity check.")



# Run Cross-Dataset Ablations

Run one dataset or all supported datasets with the same ablation protocol. Semantic experiments require `LOAD_CLIP=True` and the CLIP cell above.


In [ ]:

RUN_ABLATION = True

DATASETS_TO_RUN = ["Trento", "Houston", "MUUFL"]

PCTS = []
SHOTS = [20]
EXPERIMENTS = [
    ("baseline", None, 0.0),
    ("name", "name", 0.01),
    ("spectral", "spectral", 0.01),
    ("spectral_lidar", "spectral_lidar", 0.01),
]

EPOCHS = 200
ITERATIONS = 1
EVAL_INTERVAL = 50
SAVE_BEST_BY_TEST = False
SPECTRAL_NOISE_STD = 0.0
SPECTRAL_GAIN_STD = 0.0
FREEZE_PCT_THRESHOLD = 0.0
SEMANTIC_START_EPOCH = 50
LAMBDA_WARMUP_EPOCHS = 50

FAST_DEV_MODE = False
FAST_TEST_SAMPLES = 1024

completed_run_dirs = []

if RUN_ABLATION:
    needs_clip = any(prompt_mode is not None for _, prompt_mode, _ in EXPERIMENTS)
    for dataset in DATASETS_TO_RUN:
        print("\n" + "#" * 110)
        print(f"DATASET: {dataset}")
        load_dataset_context(dataset)
        if needs_clip:
            ensure_text_prototypes(DATASET)

        run_dir = make_run_dir(name=None)
        completed_run_dirs.append(run_dir)
        print("Run directory:", run_dir)

        if FAST_DEV_MODE:
            test_dataset, fast_counts = make_balanced_eval_subset(test_full, FAST_TEST_SAMPLES, seed=SEED)
            print("FAST_DEV_MODE: balanced test samples limited to", len(test_dataset), "counts", fast_counts)
        else:
            test_dataset = test_full

        test_loader = DataLoader(test_dataset, batch_size=TEST_BATCH_SIZE, shuffle=False)
        split_settings = [("pct", pct) for pct in PCTS] + [("shot", shot) for shot in SHOTS]
        for split_kind, split_value in split_settings:
            for iteration in range(ITERATIONS):
                split_seed = SPLIT_SEED + iteration
                if split_kind == "shot":
                    train_subset, counts = make_kshot_subset(train_full, shots=split_value, seed=split_seed)
                else:
                    train_subset, counts = make_fewshot_subset(train_full, pct=split_value, seed=split_seed)
                train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True)
                print("\n" + "=" * 100)
                print(f"{DATASET} | {split_label(split_kind, split_value)} | iteration {iteration} | samples={len(train_subset)} | counts={counts}")

                for exp_name, prompt_mode, lambda_sem in EXPERIMENTS:
                    train_one_experiment(
                        exp_name=exp_name,
                        prompt_mode=prompt_mode,
                        lambda_sem=lambda_sem,
                        train_loader=train_loader,
                        test_loader=test_loader,
                        run_dir=run_dir,
                        split_kind=split_kind,
                        split_value=split_value,
                        iteration=iteration,
                        epochs=EPOCHS,
                        eval_interval=EVAL_INTERVAL,
                        semantic_start_epoch=SEMANTIC_START_EPOCH,
                        lambda_warmup_epochs=LAMBDA_WARMUP_EPOCHS,
                        save_best_by_test=SAVE_BEST_BY_TEST,
                        spectral_noise_std=SPECTRAL_NOISE_STD,
                        spectral_gain_std=SPECTRAL_GAIN_STD,
                        freeze_pct_threshold=FREEZE_PCT_THRESHOLD,
                    )

        print("Finished dataset summary:", run_dir / "summary.csv")

    print("\nCompleted run directories:")
    for path in completed_run_dirs:
        print(" -", path)
else:
    print("RUN_ABLATION=False. Set True to run the notebook training loop.")
    print("For semantic experiments, set LOAD_CLIP=True and run the CLIP cell first.")


# Optional 1% Stabilization Study

After the main ablation if needed. This tests stronger semantic weight and spectral perturbation.


In [ ]:

RUN_STABILIZATION = False

if RUN_STABILIZATION:
    LOAD_CLIP = True
    DATASETS_TO_RUN = ["Trento", "Houston", "MUUFL"]
    PCTS = []
    SHOTS = [10]
    EXPERIMENTS = [
        ("baseline", None, 0.0),
        ("spectral_lidar", "spectral_lidar", 0.01),
        ("spectral_lidar_high_lambda", "spectral_lidar", 0.03),
    ]
    SPECTRAL_NOISE_STD = 0.01
    SPECTRAL_GAIN_STD = 0.02
    FREEZE_PCT_THRESHOLD = 0.0
    print("Now run the CLIP cell, then the ablation cell above with these settings.")
else:
    print("Stabilization setup not activated.")


# Aggregate Results


In [ ]:
# Example: RESULT_RUN_DIR = PROJECT_ROOT / "runs" / "notebook_Trento_20260526_190000"
RESULT_RUN_DIR = None

if RESULT_RUN_DIR is None:
    runs_dir = PROJECT_ROOT / "runs"
    available = sorted([p for p in runs_dir.glob("*") if (p / "summary.csv").exists()]) if runs_dir.exists() else []
    print("Available runs with summary.csv:")
    for path in available[-10:]:
        print(" -", path)
    if available:
        RESULT_RUN_DIR = available[-1]
        print("Using latest:", RESULT_RUN_DIR)

if RESULT_RUN_DIR is not None and (RESULT_RUN_DIR / "summary.csv").exists():
    summary_df = pd.read_csv(RESULT_RUN_DIR / "summary.csv")
    aggregate_df = summary_df.groupby(["pct_label", "experiment"], as_index=False).agg(
        oa_mean=("oa", "mean"),
        oa_std=("oa", "std"),
        aa_mean=("aa", "mean"),
        aa_std=("aa", "std"),
        kappa_mean=("kappa", "mean"),
        kappa_std=("kappa", "std"),
        n=("oa", "count"),
    )
    display(aggregate_df)
    out_path = RESULT_RUN_DIR / "analysis" / "aggregate_summary.csv"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    aggregate_df.to_csv(out_path, index=False)
    print("Saved:", out_path)

    base_df = summary_df[summary_df["experiment"] == "baseline"][["pct_label", "iteration", "oa", "aa", "kappa"]]
    base_df = base_df.rename(columns={"oa": "baseline_oa", "aa": "baseline_aa", "kappa": "baseline_kappa"})
    paired_df = summary_df.merge(base_df, on=["pct_label", "iteration"], how="left")
    paired_df["delta_oa"] = paired_df["oa"] - paired_df["baseline_oa"]
    paired_df["delta_aa"] = paired_df["aa"] - paired_df["baseline_aa"]
    paired_df["delta_kappa"] = paired_df["kappa"] - paired_df["baseline_kappa"]
    delta_summary_df = paired_df.groupby(["pct_label", "experiment"], as_index=False).agg(
        delta_oa_mean=("delta_oa", "mean"),
        delta_oa_std=("delta_oa", "std"),
        delta_aa_mean=("delta_aa", "mean"),
        delta_kappa_mean=("delta_kappa", "mean"),
        n=("delta_oa", "count"),
    )
    display(delta_summary_df)
    paired_path = RESULT_RUN_DIR / "analysis" / "paired_deltas.csv"
    delta_path = RESULT_RUN_DIR / "analysis" / "paired_delta_summary.csv"
    paired_df.to_csv(paired_path, index=False)
    delta_summary_df.to_csv(delta_path, index=False)
    print("Saved:", paired_path)
    print("Saved:", delta_path)
else:
    print("No completed run selected yet.")


# Plot OA Trends


In [ ]:

if RESULT_RUN_DIR is not None and (RESULT_RUN_DIR / "summary.csv").exists():
    summary_df = pd.read_csv(RESULT_RUN_DIR / "summary.csv")
    aggregate_df = summary_df.groupby(["pct_label", "experiment"], as_index=False).agg(
        oa_mean=("oa", "mean"),
        oa_std=("oa", "std"),
    )
    experiments = ["baseline", "name", "spectral", "spectral_lidar", "spectral_lidar_high_lambda"]
    styles = {
        "baseline": ("#d62728", "o", "-"),
        "name": ("#7f7f7f", "D", ":"),
        "spectral": ("#1f77b4", "s", "--"),
        "spectral_lidar": ("#2ca02c", "^", "-."),
        "spectral_lidar_high_lambda": ("#9467bd", "P", "-"),
    }
    present_pcts = sorted(set(aggregate_df["pct_label"]), key=split_sort_key)
    x = np.arange(len(present_pcts))
    plt.figure(figsize=(9, 6))
    for exp in experiments:
        rows = aggregate_df[aggregate_df["experiment"] == exp]
        if rows.empty:
            continue
        means = []
        stds = []
        for pct in present_pcts:
            row = rows[rows["pct_label"] == pct]
            means.append(np.nan if row.empty else float(row["oa_mean"].iloc[0]))
            stds.append(0.0 if row.empty or pd.isna(row["oa_std"].iloc[0]) else float(row["oa_std"].iloc[0]))
        color, marker, linestyle = styles[exp]
        plt.errorbar(x, means, yerr=stds, label=exp, color=color, marker=marker, linestyle=linestyle, capsize=5)
    plt.xticks(x, present_pcts)
    plt.xlabel("Few-shot setting")
    plt.ylabel("Overall accuracy (%)")
    plt.title("CrossHL-VLM ablation")
    plt.grid(True, linestyle="--", alpha=0.4)
    plt.legend()
    plt.tight_layout()
    fig_path = RESULT_RUN_DIR / "analysis" / "oa_trends.png"
    plt.savefig(fig_path, dpi=300)
    plt.show()
    print("Saved:", fig_path)
else:
    print("No completed run selected yet.")


# t-SNE Feature Analysis


In [ ]:

def load_checkpoint_model(checkpoint_path):
    model = CrossHL_Transformer(FM=FM, NC=NC, NCLidar=NCLIDAR, Classes=CLASSES, patchsize=PATCH_SIZE).to(DEVICE)
    state = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(state)
    model.eval()
    return model


def extract_cls_features(model, dataset, batch_size=500):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    features = []
    labels = []
    with torch.no_grad():
        for hsi_batch, lidar_batch, label_batch in loader:
            hsi_batch = hsi_batch.to(DEVICE)
            lidar_batch = lidar_batch.to(DEVICE)
            _, cls = model(hsi_batch, lidar_batch, return_features=True)
            features.append(cls.cpu().numpy())
            labels.append(label_batch.numpy())
    return np.concatenate(features), np.concatenate(labels)


def plot_tsne_for_run(run_dir, pct_label="20-shot", iteration=0, num_samples=1500):
    summary_df = pd.read_csv(run_dir / "summary.csv")
    dataset = summary_df["dataset"].iloc[0]
    load_dataset_context(dataset)
    experiments = ["baseline", "name", "spectral", "spectral_lidar"]
    fig, axes = plt.subplots(1, len(experiments), figsize=(6 * len(experiments), 5))
    sample_size = min(num_samples, len(test_full))
    sample_idx = np.random.default_rng(42).choice(len(test_full), sample_size, replace=False)

    for ax, exp in zip(axes, experiments):
        row = summary_df[(summary_df["pct_label"] == pct_label) & (summary_df["iteration"] == iteration) & (summary_df["experiment"] == exp)]
        if row.empty:
            ax.set_title(f"Missing {exp}")
            ax.axis("off")
            continue
        checkpoint = Path(row["checkpoint"].iloc[0])
        model = load_checkpoint_model(checkpoint)
        features, labels = extract_cls_features(model, test_full)
        embedded = TSNE(n_components=2, random_state=42, init="pca", perplexity=min(30, max(5, sample_size // 20))).fit_transform(features[sample_idx])
        sampled_labels = labels[sample_idx]
        for class_id, class_name in enumerate(class_names):
            mask = sampled_labels == class_id
            ax.scatter(embedded[mask, 0], embedded[mask, 1], s=16, alpha=0.65, label=class_name)
        ax.set_title(exp)
        ax.axis("off")

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=min(len(class_names), 6))
    fig.suptitle(f"{dataset} CLS feature t-SNE, {pct_label}, iteration {iteration}")
    plt.tight_layout(rect=[0, 0.08, 1, 0.95])
    safe_label = pct_label.replace("%", "pct").replace(" ", "_")
    fig_path = run_dir / "analysis" / f"tsne_cls_{safe_label}_iter{iteration}.png"
    plt.savefig(fig_path, dpi=300)
    plt.show()
    print("Saved:", fig_path)

RUN_TSNE = False
if RUN_TSNE and RESULT_RUN_DIR is not None:
    plot_tsne_for_run(RESULT_RUN_DIR, pct_label="20-shot", iteration=0)
else:
    print("Set RUN_TSNE=True after a completed run.")


# Extended Evaluation Showcase

This section generates extended evaluation outputs:

1. Generate interpretability heatmaps for real test patches from the selected checkpoint.
2. Produce statistical reports beyond OA/AA/kappa: MCC, macro-F1, micro-F1, log loss, ECE, per-class report, and confusion matrices.
3. Show boundary/interior behavior. The current patch datasets contain vector labels but not full spatial coordinates, so the runnable cell below provides a clearly labeled hard-patch/boundary-like proxy. Use a true full-scene ground-truth map later when available.
4. Prepare a cleaner architecture figure concept that emphasizes the dual branch after the fused CLS feature.

Interpretation:

> The first Trento results show that the Cross-HL baseline is strong. The notebook therefore extends the evaluation beyond OA/AA/kappa to include statistical generalization, calibration, patch-level interpretability, and boundary-like/hard-region behavior.

## Evaluation Statistics: MCC, F1, Log Loss, ECE, Confusion Matrix

In [ ]:
def available_checkpoint_rows(run_dir):
    run_dir = Path(run_dir)
    summary_path = run_dir / "summary.csv"
    if not summary_path.exists():
        raise FileNotFoundError(f"Missing summary.csv: {summary_path}")
    df = pd.read_csv(summary_path)
    df["iteration"] = df["iteration"].astype(int)
    return df[["dataset", "pct_label", "experiment", "iteration", "checkpoint"]].drop_duplicates().reset_index(drop=True)


def resolve_showcase_split(run_dir, preferred_split=None, iteration=0):
    rows = available_checkpoint_rows(run_dir)
    rows = rows[rows["iteration"] == int(iteration)]
    if rows.empty:
        rows = available_checkpoint_rows(run_dir)
    available_splits = sorted(rows["pct_label"].astype(str).unique().tolist(), key=split_sort_key)
    if not available_splits:
        raise ValueError("No available splits in summary.csv")
    if preferred_split is not None and str(preferred_split) in available_splits:
        return str(preferred_split)
    chosen = available_splits[0]
    if preferred_split is not None:
        print(f"Requested split={preferred_split!r} is not in this run. Using available split={chosen!r}.")
    return chosen


def resolve_showcase_experiments(run_dir, preferred_experiments=None, split=None, iteration=0):
    rows = available_checkpoint_rows(run_dir)
    rows = rows[rows["iteration"] == int(iteration)]
    if split is not None:
        rows = rows[rows["pct_label"].astype(str) == str(split)]
    available = rows["experiment"].astype(str).unique().tolist()
    if preferred_experiments is None:
        preferred_experiments = ["baseline", "spectral_lidar"]
    chosen = [exp for exp in preferred_experiments if exp in available]
    if not chosen:
        chosen = available
    missing = [exp for exp in preferred_experiments if exp not in available]
    if missing:
        print("Skipping missing experiments in this run:", missing)
        print("Available experiments:", available)
    return chosen


def select_checkpoint_row(run_dir, experiment="spectral_lidar", split=None, iteration=0):
    """Return one summary row and checkpoint path from the current run structure.

    Use split=None to automatically select the available split for quick 1-iteration runs.
    """
    run_dir = Path(run_dir)
    df = available_checkpoint_rows(run_dir)
    df["_experiment"] = df["experiment"].astype(str)
    df["_split"] = df["pct_label"].astype(str)

    if split is None or str(split).lower() == "auto":
        split = resolve_showcase_split(run_dir, preferred_split=None, iteration=iteration)

    mask = (
        (df["_experiment"] == str(experiment))
        & (df["_split"] == str(split))
        & (df["iteration"].astype(int) == int(iteration))
    )
    if not mask.any():
        fallback = df[(df["_experiment"] == str(experiment)) & (df["iteration"].astype(int) == int(iteration))]
        if not fallback.empty:
            row = fallback.iloc[0]
            print(
                f"Requested split={split!r} was not found for experiment={experiment!r}. "
                f"Using available split={row['pct_label']!r}."
            )
        else:
            available = df[["dataset", "pct_label", "experiment", "iteration", "checkpoint"]].drop_duplicates()
            print("Available checkpoint rows:")
            display(available.tail(30))
            raise ValueError(f"No checkpoint found for experiment={experiment}, split={split}, iteration={iteration}")
    else:
        row = df[mask].iloc[0]

    ckpt = Path(row["checkpoint"])
    if not ckpt.exists():
        ckpt = PROJECT_ROOT / ckpt
    if not ckpt.exists():
        raise FileNotFoundError(f"Checkpoint path in summary.csv does not exist: {row['checkpoint']}")
    return row, ckpt


def load_model_from_checkpoint(checkpoint_path, dataset=None):
    """Load CrossHL with dimensions inferred from the selected dataset."""
    if dataset is not None and dataset != DATASET:
        load_dataset_context(dataset)
    model = CrossHL_Transformer(
        FM=FM,
        NC=NC,
        NCLidar=NCLIDAR,
        Classes=CLASSES,
        patchsize=PATCH_SIZE,
    ).to(DEVICE)
    state = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(state)
    model.eval()
    return model


def predict_probabilities(model, dataset, batch_size=500):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    y_true, y_pred, probs = [], [], []
    model.eval()
    with torch.no_grad():
        for hsi_batch, lidar_batch, label_batch in loader:
            logits = model(hsi_batch.to(DEVICE), lidar_batch.to(DEVICE))
            prob = torch.softmax(logits, dim=1)
            y_true.append(label_batch.cpu().numpy())
            y_pred.append(prob.argmax(dim=1).cpu().numpy())
            probs.append(prob.cpu().numpy())
    return np.concatenate(y_true), np.concatenate(y_pred), np.concatenate(probs)


def expected_calibration_error(y_true, probs, n_bins=15):
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    accuracies = predictions == y_true
    ece = 0.0
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        in_bin = (confidences >= lo) & (confidences < hi)
        if not np.any(in_bin):
            continue
        bin_acc = accuracies[in_bin].mean()
        bin_conf = confidences[in_bin].mean()
        ece += np.abs(bin_acc - bin_conf) * in_bin.mean()
    return float(ece)


def latest_run_dirs_by_dataset(preferred_experiments=None):
    """Find latest run directory for each dataset, preferring runs with requested experiments."""
    preferred = None if preferred_experiments is None else set(map(str, preferred_experiments))
    preferred_found = {}
    fallback_found = {}
    for summary_path in (PROJECT_ROOT / "runs").glob("*/summary.csv"):
        try:
            df = pd.read_csv(summary_path)
            dataset = str(df["dataset"].iloc[0])
            experiments = set(df["experiment"].astype(str).unique())
        except Exception:
            continue
        run_dir = summary_path.parent
        mtime = run_dir.stat().st_mtime
        if dataset not in fallback_found or mtime > fallback_found[dataset][0]:
            fallback_found[dataset] = (mtime, run_dir, experiments)
        if preferred is None or preferred.issubset(experiments):
            if dataset not in preferred_found or mtime > preferred_found[dataset][0]:
                preferred_found[dataset] = (mtime, run_dir, experiments)
    selected = {}
    for dataset in sorted(set(fallback_found) | set(preferred_found)):
        selected[dataset] = (preferred_found.get(dataset) or fallback_found[dataset])[1]
    return selected


def print_selected_run_dirs(run_dirs):
    for dataset, run_dir in run_dirs.items() if isinstance(run_dirs, dict) else enumerate(run_dirs):
        print(f"{dataset}: {run_dir}")


def plot_confusion_matrix(cm, names, title, save_path):
    cm = np.asarray(cm)
    denom = cm.sum(axis=1, keepdims=True)
    cm_norm = np.divide(cm, np.maximum(denom, 1), where=np.maximum(denom, 1) != 0)
    fig, ax = plt.subplots(figsize=(max(7, len(names) * 0.75), max(6, len(names) * 0.65)))
    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(np.arange(len(names)))
    ax.set_yticks(np.arange(len(names)))
    ax.set_xticklabels(names, rotation=45, ha="right")
    ax.set_yticklabels(names)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    for i in range(cm_norm.shape[0]):
        for j in range(cm_norm.shape[1]):
            color = "white" if cm_norm[i, j] > 0.55 else "black"
            ax.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center", color=color, fontsize=8)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", save_path)


def full_statistical_report_for_checkpoint(run_dir, experiment="spectral_lidar", split=None, iteration=0):
    """Extended evaluation statistics: MCC, F1, log loss, ECE, per-class report, confusion matrix."""
    row, ckpt = select_checkpoint_row(run_dir, experiment=experiment, split=split, iteration=iteration)
    dataset = row["dataset"]
    split = row["pct_label"]
    load_dataset_context(dataset)
    model = load_model_from_checkpoint(ckpt, dataset=dataset)
    y_true, y_pred, probs = predict_probabilities(model, test_full, batch_size=TEST_BATCH_SIZE)

    labels = list(range(CLASSES))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    metrics = {
        "dataset": dataset,
        "experiment": experiment,
        "split": split,
        "iteration": int(iteration),
        "oa": accuracy_score(y_true, y_pred) * 100.0,
        "kappa": cohen_kappa_score(y_true, y_pred) * 100.0,
        "mcc": matthews_corrcoef(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "micro_f1": f1_score(y_true, y_pred, average="micro"),
        "log_loss": log_loss(y_true, probs, labels=labels),
        "ece": expected_calibration_error(y_true, probs),
        "checkpoint": str(ckpt),
    }
    report = classification_report(y_true, y_pred, target_names=class_names, digits=4, zero_division=0)

    out_dir = Path(run_dir) / "analysis" / "extended_evaluation"
    out_dir.mkdir(parents=True, exist_ok=True)
    safe_split = str(split).replace("%", "pct").replace(" ", "_")
    tag = f"{dataset}_{experiment}_{safe_split}_iter{iteration}"
    pd.DataFrame([metrics]).to_csv(out_dir / f"stats_{tag}.csv", index=False)
    (out_dir / f"classification_report_{tag}.txt").write_text(report, encoding="utf-8")
    plot_confusion_matrix(cm, class_names, f"{dataset} {experiment} {split} iter {iteration}", out_dir / f"confusion_{tag}.png")

    print("\nMetrics")
    display(pd.DataFrame([metrics]).drop(columns=["checkpoint"]))
    print("\nPer-class report")
    print(report)
    return metrics, report


def statistical_reports_for_many_runs(run_dirs=None, split=None, iteration=0, preferred_experiments=None):
    preferred = preferred_experiments or ["baseline", "spectral_lidar"]
    if run_dirs is None:
        run_dirs = latest_run_dirs_by_dataset(preferred_experiments=preferred)
    if isinstance(run_dirs, dict):
        run_dirs = list(run_dirs.values())
    print("Selected run directories:")
    for run_dir in run_dirs:
        print(" -", run_dir)

    all_metrics = []
    for run_dir in run_dirs:
        print("\n" + "=" * 100)
        print("Statistics from:", run_dir)
        active_split = resolve_showcase_split(run_dir, preferred_split=split, iteration=iteration)
        active_experiments = resolve_showcase_experiments(
            run_dir,
            preferred_experiments=preferred,
            split=active_split,
            iteration=iteration,
        )
        print("Using split:", active_split)
        print("Using experiments:", active_experiments)
        for exp in active_experiments:
            metrics, _ = full_statistical_report_for_checkpoint(run_dir, experiment=exp, split=active_split, iteration=iteration)
            all_metrics.append(metrics)

    out = pd.DataFrame(all_metrics)
    if not out.empty:
        out_dir = PROJECT_ROOT / "runs" / "extended_evaluation_all_datasets"
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / "all_dataset_statistics.csv"
        out.to_csv(out_path, index=False)
        print("\nCombined statistics:")
        display(out.drop(columns=["checkpoint"], errors="ignore"))
        print("Saved:", out_path)
    return out


SHOWCASE_ITERATION = 0
SHOWCASE_SPLIT = None  # None means auto-select, e.g. 20-shot for the current fast runs.
SHOWCASE_EXPERIMENTS = None  # None means baseline+spectral_lidar when present; otherwise available experiments.
SHOWCASE_RUN_DIRS = None  # None means auto-select latest suitable Trento/Houston/MUUFL runs.

RUN_STAT_REPORT = False
RUN_STAT_REPORT_ALL_DATASETS = False
if RUN_STAT_REPORT and RESULT_RUN_DIR is not None:
    active_split = resolve_showcase_split(RESULT_RUN_DIR, preferred_split=SHOWCASE_SPLIT, iteration=SHOWCASE_ITERATION)
    active_experiments = resolve_showcase_experiments(
        RESULT_RUN_DIR,
        preferred_experiments=SHOWCASE_EXPERIMENTS,
        split=active_split,
        iteration=SHOWCASE_ITERATION,
    )
    print("Using split:", active_split)
    print("Using experiments:", active_experiments)
    for exp in active_experiments:
        full_statistical_report_for_checkpoint(RESULT_RUN_DIR, experiment=exp, split=active_split, iteration=SHOWCASE_ITERATION)
elif RUN_STAT_REPORT_ALL_DATASETS:
    statistical_reports_for_many_runs(
        run_dirs=SHOWCASE_RUN_DIRS,
        split=SHOWCASE_SPLIT,
        iteration=SHOWCASE_ITERATION,
        preferred_experiments=SHOWCASE_EXPERIMENTS,
    )
else:
    print("Set RUN_STAT_REPORT=True for one run, or RUN_STAT_REPORT_ALL_DATASETS=True for Trento/Houston/MUUFL.")


## Interpretability Heatmaps: Conv Attribution on HSI Patch + LiDAR View

In [ ]:
def normalize_image(x, eps=1e-8, percentile=True):
    x = np.asarray(x, dtype=np.float32)
    if percentile:
        lo, hi = np.percentile(x, [2, 98])
    else:
        lo, hi = x.min(), x.max()
    return np.clip((x - lo) / (hi - lo + eps), 0, 1)


def smooth_map(x, sigma=0.8):
    try:
        from scipy.ndimage import gaussian_filter
        return gaussian_filter(x, sigma=sigma)
    except Exception:
        return x


def false_color_hsi(hsi_patch):
    """Make a stable false-color view for any HSI band count.

    This is not natural RGB. It is only a display composite from three HSI bands.
    """
    hsi_np = hsi_patch.detach().cpu().numpy()
    bands = hsi_np.shape[0]
    idx = [int(0.70 * (bands - 1)), int(0.45 * (bands - 1)), int(0.20 * (bands - 1))]
    channels = [normalize_image(hsi_np[i], percentile=True) for i in idx]
    return np.stack(channels, axis=-1)


def lidar_view(lidar_patch):
    lidar_np = lidar_patch.detach().cpu().numpy()
    if lidar_np.ndim == 3:
        lidar_np = lidar_np.mean(axis=0)
    return normalize_image(lidar_np, percentile=True)


def conv_gradcam(model, hsi_patch, lidar_patch, target_class=None):
    """Grad-CAM-style attribution on Cross-HL's last HetConv groupwise conv layer."""
    model.eval()
    target_layer = model.hetconv_layer[0].groupwise_conv
    activations = {}
    gradients = {}

    def forward_hook(module, inputs, output):
        activations["value"] = output.detach()

    def backward_hook(module, grad_input, grad_output):
        gradients["value"] = grad_output[0].detach()

    handle_fwd = target_layer.register_forward_hook(forward_hook)
    handle_bwd = target_layer.register_full_backward_hook(backward_hook)
    try:
        hsi = hsi_patch.unsqueeze(0).to(DEVICE)
        lidar = lidar_patch.unsqueeze(0).to(DEVICE)
        model.zero_grad(set_to_none=True)
        logits = model(hsi, lidar)
        pred = int(logits.argmax(dim=1).item())
        cls = pred if target_class is None else int(target_class)
        score = logits[:, cls].sum()
        score.backward()

        acts = activations["value"][0]
        grads = gradients["value"][0]
        weights = grads.mean(dim=(1, 2), keepdim=True)
        cam = torch.relu((weights * acts).sum(dim=0))
        cam = cam.detach().cpu().numpy()
        cam = smooth_map(cam, sigma=0.8)
        cam = normalize_image(cam, percentile=False)
        probs = torch.softmax(logits.detach(), dim=1).squeeze(0).cpu().numpy()
    finally:
        handle_fwd.remove()
        handle_bwd.remove()
    return cam, pred, probs


def choose_representative_test_indices(model, dataset, max_classes=None, prefer_correct=True):
    """Choose high-confidence samples so the visual examples are more presentable."""
    y_true, y_pred, probs = predict_probabilities(model, dataset, batch_size=TEST_BATCH_SIZE)
    conf = probs.max(axis=1)
    indices = []
    for c in range(CLASSES):
        class_idx = np.where(y_true == c)[0]
        if len(class_idx) == 0:
            continue
        if prefer_correct:
            candidate_idx = class_idx[y_pred[class_idx] == c]
            if len(candidate_idx) == 0:
                candidate_idx = class_idx
        else:
            candidate_idx = class_idx
        best = int(candidate_idx[np.argmax(conf[candidate_idx])])
        indices.append(best)
        if max_classes is not None and len(indices) >= max_classes:
            break
    return indices


def plot_decision_heatmap(model, dataset, index, save_path=None, target_class=None, title_prefix=""):
    hsi_patch, lidar_patch, label = dataset[index]
    cam, pred, probs = conv_gradcam(model, hsi_patch, lidar_patch, target_class=target_class)
    true_label = int(label.item())
    rgb = false_color_hsi(hsi_patch)
    lidar_img = lidar_view(lidar_patch)

    fig, axes = plt.subplots(1, 4, figsize=(17, 4.5))
    axes[0].imshow(rgb, interpolation="nearest")
    axes[0].set_title(f"HSI false-color patch\nTrue: {class_names[true_label]}")
    axes[0].axis("off")

    axes[1].imshow(rgb, interpolation="nearest")
    axes[1].imshow(cam, cmap="jet", alpha=0.48, interpolation="bilinear")
    axes[1].set_title(f"Conv attribution overlay\nPred: {class_names[pred]}")
    axes[1].axis("off")

    im = axes[2].imshow(lidar_img, cmap="terrain", interpolation="nearest")
    axes[2].set_title("LiDAR patch view")
    axes[2].axis("off")
    fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

    order = np.argsort(probs)[::-1][:min(8, len(probs))]
    y = np.arange(len(order))
    colors = ["tab:green" if i == true_label else "tab:blue" for i in order]
    axes[3].barh(y, probs[order], color=colors)
    axes[3].set_yticks(y)
    axes[3].set_yticklabels([class_names[i] for i in order])
    axes[3].invert_yaxis()
    axes[3].set_xlim(0, 1)
    axes[3].set_title("Top probabilities")

    correct = "correct" if pred == true_label else "wrong"
    fig.suptitle(f"{title_prefix} sample {index} - {correct}", fontsize=13)
    plt.tight_layout()
    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        print("Saved:", save_path)
    plt.show()
    return {"index": int(index), "true": true_label, "pred": pred, "confidence": float(probs[pred])}


def generate_heatmaps_for_run(run_dir, experiment="spectral_lidar", split=None, iteration=0, max_classes=6):
    row, ckpt = select_checkpoint_row(run_dir, experiment=experiment, split=split, iteration=iteration)
    dataset = row["dataset"]
    split = row["pct_label"]
    load_dataset_context(dataset)
    model = load_model_from_checkpoint(ckpt, dataset=dataset)
    out_dir = Path(run_dir) / "analysis" / "extended_evaluation" / "heatmaps" / f"{experiment}_{str(split).replace('%','pct')}_iter{iteration}"
    rows = []
    indices = choose_representative_test_indices(model, test_full, max_classes=max_classes, prefer_correct=True)
    for idx in indices:
        true_label = int(test_full.lbls[idx].item())
        save_path = out_dir / f"class_{true_label:02d}_{class_names[true_label].replace(' ', '_')}_idx{idx}.png"
        rows.append(plot_decision_heatmap(model, test_full, idx, save_path=save_path, title_prefix=f"{dataset} {experiment} {split}"))
    result = pd.DataFrame(rows)
    result.to_csv(out_dir / "heatmap_samples.csv", index=False)
    display(result)
    return result


def latest_run_dirs_by_dataset(preferred_experiments=None):
    """Find latest run directory for each dataset, preferring runs with requested experiments."""
    preferred = None if preferred_experiments is None else set(map(str, preferred_experiments))
    preferred_found = {}
    fallback_found = {}
    for summary_path in (PROJECT_ROOT / "runs").glob("*/summary.csv"):
        try:
            df = pd.read_csv(summary_path)
            dataset = str(df["dataset"].iloc[0])
            experiments = set(df["experiment"].astype(str).unique())
        except Exception:
            continue
        run_dir = summary_path.parent
        mtime = run_dir.stat().st_mtime
        if dataset not in fallback_found or mtime > fallback_found[dataset][0]:
            fallback_found[dataset] = (mtime, run_dir, experiments)
        if preferred is None or preferred.issubset(experiments):
            if dataset not in preferred_found or mtime > preferred_found[dataset][0]:
                preferred_found[dataset] = (mtime, run_dir, experiments)
    selected = {}
    for dataset in sorted(set(fallback_found) | set(preferred_found)):
        selected[dataset] = (preferred_found.get(dataset) or fallback_found[dataset])[1]
    return selected


def generate_heatmaps_for_many_runs(run_dirs=None, experiment="spectral_lidar", split=None, iteration=0, max_classes=3):
    if run_dirs is None:
        run_dirs = list(latest_run_dirs_by_dataset(preferred_experiments=[experiment]).values())
    for run_dir in run_dirs:
        print("\n" + "=" * 100)
        print("Heatmaps from:", run_dir)
        rows = available_checkpoint_rows(run_dir)
        active_split = resolve_showcase_split(run_dir, preferred_split=split, iteration=iteration)
        active_experiments = resolve_showcase_experiments(
            run_dir,
            preferred_experiments=[experiment, "spectral", "baseline"],
            split=active_split,
            iteration=iteration,
        )
        active_exp = experiment if experiment in active_experiments else active_experiments[0]
        generate_heatmaps_for_run(run_dir, experiment=active_exp, split=active_split, iteration=iteration, max_classes=max_classes)


RUN_HEATMAPS = False
RUN_HEATMAPS_ALL_DATASETS = False
if RUN_HEATMAPS and RESULT_RUN_DIR is not None:
    active_split = resolve_showcase_split(RESULT_RUN_DIR, preferred_split=SHOWCASE_SPLIT, iteration=SHOWCASE_ITERATION)
    active_experiments = resolve_showcase_experiments(
        RESULT_RUN_DIR,
        preferred_experiments=["spectral_lidar", "spectral", "baseline"],
        split=active_split,
        iteration=SHOWCASE_ITERATION,
    )
    heatmap_exp = "spectral_lidar" if "spectral_lidar" in active_experiments else active_experiments[0]
    print("Using split:", active_split)
    print("Using heatmap experiment:", heatmap_exp)
    generate_heatmaps_for_run(RESULT_RUN_DIR, experiment=heatmap_exp, split=active_split, iteration=SHOWCASE_ITERATION, max_classes=6)
elif RUN_HEATMAPS_ALL_DATASETS:
    generate_heatmaps_for_many_runs(SHOWCASE_RUN_DIRS, experiment="spectral_lidar", split=SHOWCASE_SPLIT, iteration=SHOWCASE_ITERATION, max_classes=3)
else:
    print("Set RUN_HEATMAPS=True for the selected RESULT_RUN_DIR, or RUN_HEATMAPS_ALL_DATASETS=True for latest runs across datasets.")


## Boundary / Center Behavior

In [ ]:
def patch_heterogeneity_scores(dataset):
    """Proxy for mixed/boundary-like patches using spatial variability inside the patch."""
    hsi = dataset.hs_image.float()
    lidar = dataset.lidar_image.float()
    hsi_score = hsi.var(dim=(2, 3)).mean(dim=1).detach().cpu().numpy()
    lidar_score = lidar.var(dim=(2, 3)).mean(dim=1).detach().cpu().numpy()
    hsi_score = normalize_image(hsi_score)
    lidar_score = normalize_image(lidar_score)
    return 0.75 * hsi_score + 0.25 * lidar_score


def hard_patch_analysis_for_checkpoint(run_dir, experiment="spectral_lidar", split=None, iteration=0, quantile=0.25):
    """Runnable proxy for the boundary/center analysis.

    True boundary-vs-center requires a full spatial GT map plus coordinates for each patch.
    The current released patch files contain vector labels only, so this function reports:
    1) low-confidence hard patches vs the rest;
    2) high-heterogeneity boundary-like patches vs low-heterogeneity center-like patches.
    """
    row, ckpt = select_checkpoint_row(run_dir, experiment=experiment, split=split, iteration=iteration)
    dataset = row["dataset"]
    split = row["pct_label"]
    load_dataset_context(dataset)
    model = load_model_from_checkpoint(ckpt, dataset=dataset)
    y_true, y_pred, probs = predict_probabilities(model, test_full, batch_size=TEST_BATCH_SIZE)
    correct = y_pred == y_true
    confidence = probs.max(axis=1)
    heterogeneity = patch_heterogeneity_scores(test_full)

    low_conf = confidence <= np.quantile(confidence, quantile)
    high_conf = confidence >= np.quantile(confidence, 1.0 - quantile)
    high_het = heterogeneity >= np.quantile(heterogeneity, 1.0 - quantile)
    low_het = heterogeneity <= np.quantile(heterogeneity, quantile)

    rows = [
        {"subset": "low-confidence hard patches", "n": int(low_conf.sum()), "accuracy": float(correct[low_conf].mean() * 100.0), "mean_confidence": float(confidence[low_conf].mean())},
        {"subset": "high-confidence easy patches", "n": int(high_conf.sum()), "accuracy": float(correct[high_conf].mean() * 100.0), "mean_confidence": float(confidence[high_conf].mean())},
        {"subset": "high-heterogeneity boundary-like patches", "n": int(high_het.sum()), "accuracy": float(correct[high_het].mean() * 100.0), "mean_confidence": float(confidence[high_het].mean())},
        {"subset": "low-heterogeneity center-like patches", "n": int(low_het.sum()), "accuracy": float(correct[low_het].mean() * 100.0), "mean_confidence": float(confidence[low_het].mean())},
    ]
    out = pd.DataFrame(rows)
    out_dir = Path(run_dir) / "analysis" / "extended_evaluation"
    out_dir.mkdir(parents=True, exist_ok=True)
    safe_split = str(split).replace("%", "pct").replace(" ", "_")
    out.to_csv(out_dir / f"hard_patch_proxy_{dataset}_{experiment}_{safe_split}_iter{iteration}.csv", index=False)

    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.bar(out["subset"], out["accuracy"], color=["tab:red", "tab:green", "tab:orange", "tab:blue"])
    ax.set_ylabel("Accuracy (%)")
    ax.set_ylim(0, 100)
    ax.set_title(f"{dataset} {experiment} {split}: hard-patch / boundary-like proxy")
    ax.tick_params(axis="x", rotation=25)
    plt.tight_layout()
    fig_path = out_dir / f"hard_patch_proxy_{dataset}_{experiment}_{safe_split}_iter{iteration}.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", fig_path)
    out.insert(0, "split", split)
    out.insert(0, "experiment", experiment)
    out.insert(0, "dataset", dataset)
    display(out)
    print("Note: this is a proxy. For the exact boundary-center test, add full-scene GT coordinates for each patch.")
    return out


def hard_patch_analysis_for_many_runs(run_dirs=None, split=None, iteration=0, preferred_experiments=None):
    preferred = preferred_experiments or ["baseline", "spectral_lidar"]
    if run_dirs is None:
        run_dirs = latest_run_dirs_by_dataset(preferred_experiments=preferred)
    if isinstance(run_dirs, dict):
        run_dirs = list(run_dirs.values())
    print("Selected run directories:")
    for run_dir in run_dirs:
        print(" -", run_dir)

    combined = []
    for run_dir in run_dirs:
        print("\n" + "=" * 100)
        print("Hard-patch proxy from:", run_dir)
        active_split = resolve_showcase_split(run_dir, preferred_split=split, iteration=iteration)
        active_experiments = resolve_showcase_experiments(
            run_dir,
            preferred_experiments=preferred,
            split=active_split,
            iteration=iteration,
        )
        print("Using split:", active_split)
        print("Using experiments:", active_experiments)
        for exp in active_experiments:
            combined.append(hard_patch_analysis_for_checkpoint(run_dir, experiment=exp, split=active_split, iteration=iteration))

    out = pd.concat(combined, ignore_index=True) if combined else pd.DataFrame()
    if not out.empty:
        out_dir = PROJECT_ROOT / "runs" / "extended_evaluation_all_datasets"
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / "all_dataset_hard_patch_proxy.csv"
        out.to_csv(out_path, index=False)
        print("\nCombined hard-patch proxy:")
        display(out)
        print("Saved:", out_path)
    return out


RUN_HARD_PATCH_ANALYSIS = False
RUN_HARD_PATCH_ANALYSIS_ALL_DATASETS = False
if RUN_HARD_PATCH_ANALYSIS and RESULT_RUN_DIR is not None:
    active_split = resolve_showcase_split(RESULT_RUN_DIR, preferred_split=SHOWCASE_SPLIT, iteration=SHOWCASE_ITERATION)
    active_experiments = resolve_showcase_experiments(
        RESULT_RUN_DIR,
        preferred_experiments=SHOWCASE_EXPERIMENTS,
        split=active_split,
        iteration=SHOWCASE_ITERATION,
    )
    print("Using split:", active_split)
    print("Using experiments:", active_experiments)
    for exp in active_experiments:
        hard_patch_analysis_for_checkpoint(RESULT_RUN_DIR, experiment=exp, split=active_split, iteration=SHOWCASE_ITERATION)
elif RUN_HARD_PATCH_ANALYSIS_ALL_DATASETS:
    hard_patch_analysis_for_many_runs(
        run_dirs=SHOWCASE_RUN_DIRS,
        split=SHOWCASE_SPLIT,
        iteration=SHOWCASE_ITERATION,
        preferred_experiments=SHOWCASE_EXPERIMENTS,
    )
else:
    print("Set RUN_HARD_PATCH_ANALYSIS=True for one run, or RUN_HARD_PATCH_ANALYSIS_ALL_DATASETS=True for Trento/Houston/MUUFL.")


# Architecture Figure Specification

The final architecture figure should follow this specification and be redrawn manually in a paper-ready style using TikZ, draw.io, or PowerPoint.

> Draw a professional neural-network architecture diagram for a paper. The model takes a native HSI patch and a LiDAR patch as inputs. They go through a Cross-HL encoder for HSI+LiDAR feature fusion. The fused CLS feature splits into two branches: one branch goes to the original classifier head with cross-entropy loss and produces the final inference prediction; the second branch goes through a small projection MLP to produce an L2-normalized student embedding. This student embedding is aligned with frozen CLIP text prototypes generated from physics-guided spectral and LiDAR prompts using a contrastive semantic loss. The CLIP image encoder is not used, CLIP is frozen, and final prediction comes only from the Cross-HL classifier.

Figure requirements:

- Make the dual branch after the fused CLS feature visually central.
- Mark CLIP text encoder as frozen.
- Show that CLIP gives training regularization only.
- Do not show HSI-to-RGB conversion.
- Do not show CLIP image encoder.
- At inference, arrow should end at classifier prediction, not at CLIP.

# Legacy Result Tables

Load archived result tables when available.


In [ ]:
legacy_dir = PROJECT_ROOT / "results" / "legacy"
for filename in ["Master_Paper_Results.csv", "Master_PerClass_Accuracy.csv"]:
    path = legacy_dir / filename
    if path.exists():
        print("\n" + filename)
        display(pd.read_csv(path))
    else:
        print("Missing:", path)
